# 1️⃣ BLOQUE 1 — DATASET (PDF → TEXTO)

En este primer bloque se construye la base documental del sistema RAG.

**Qué se hace aquí**
- Se prepara la carpeta de trabajo del proyecto.
- Se descargan documentos PDF reales del dominio de encuestas.
- Se extrae el texto página por página para convertirlo en una colección procesable.

**Por qué es importante**
Un sistema RAG necesita una fuente de conocimiento externa. En este caso, el corpus está formado por manuales y documentos metodológicos del INEI y del World Bank, lo que permite responder preguntas sobre encuestas usando información real y verificable.

In [ ]:
import os

BASE_DIR = "/content/rag_encuestas"
PDF_DIR = os.path.join(BASE_DIR, "pdfs")

os.makedirs(PDF_DIR, exist_ok=True)

print("Carpetas creadas en:", BASE_DIR)

Carpetas creadas en: /content/rag_encuestas


## Celda 2 — Descargar PDFs reales

**Objetivo:** construir el corpus inicial del sistema.

**Qué hace esta celda**
- Define un conjunto de URLs con documentos fuente.
- Descarga cada PDF a la carpeta local del proyecto.
- Verifica qué archivos quedaron disponibles.

**Interpretación**
Aquí se realiza la **ingesta documental**. Esta etapa es clave porque el sistema no depende de conocimiento memorizado del modelo, sino de documentos externos que luego serán indexados y consultados.

In [ ]:
import requests

pdf_urls = {
"manual_encuestador_inei.pdf":
"https://proyectos.inei.gob.pe/iinei/srienaho/Descarga/DocumentosMetodologicos/2022-55/13_Manual_del_Encuestador.pdf",

"buenas_practicas_muestreo_inei.pdf":
"https://www.inei.gob.pe/media/MenuRecursivo/metodologias/encuestas01.pdf",

"enterprise_surveys_manual_worldbank.pdf":
"https://www.worldbank.org/content/dam/enterprisesurveys/documents/methodology/Enterprise%20Surveys_Manual%20and%20Guide.pdf"
}

for filename, url in pdf_urls.items():

    out_path = os.path.join(PDF_DIR, filename)

    r = requests.get(url, timeout=60)
    r.raise_for_status()

    with open(out_path, "wb") as f:
        f.write(r.content)

    print("Descargado:", filename)

print("\nPDFs disponibles:")
print(os.listdir(PDF_DIR))

Descargado: manual_encuestador_inei.pdf
Descargado: buenas_practicas_muestreo_inei.pdf
Descargado: enterprise_surveys_manual_worldbank.pdf

PDFs disponibles:
['manual_encuestador_inei.pdf', 'enterprise_surveys_manual_worldbank.pdf', 'buenas_practicas_muestreo_inei.pdf']


## Celda 3 — Extraer texto de los PDFs

**Objetivo:** convertir documentos PDF en texto utilizable.

**Qué hace esta celda**
- Usa `pypdf` para leer cada archivo.
- Recorre sus páginas.
- Extrae texto y lo almacena junto con el nombre del archivo y el número de página.

**Interpretación**
Esta etapa transforma documentos estáticos en una colección estructurada. Cada página pasa a ser una unidad intermedia que luego podrá dividirse en fragmentos más pequeños para recuperación eficiente.

In [ ]:
!pip install pypdf

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)
    pages = []

    for i, page in enumerate(reader.pages):

        text = page.extract_text()

        if text:
            pages.append({
                "page_num": i+1,
                "text": text
            })

    return pages


documents = []

for pdf_file in os.listdir(PDF_DIR):

    if pdf_file.lower().endswith(".pdf"):

        pdf_path = os.path.join(PDF_DIR, pdf_file)

        pages = extract_text_from_pdf(pdf_path)

        for page in pages:

            documents.append({
                "source_file": pdf_file,
                "page_num": page["page_num"],
                "text": page["text"]
            })

print("Total páginas extraídas:", len(documents))

Total páginas extraídas: 492


# 2️⃣ BLOQUE 2 — CHUNKING

## Celda 4 — Limpiar texto

**Objetivo:** normalizar el texto antes del chunking.

**Qué hace esta celda**
- Elimina saltos de línea, tabulaciones y espacios múltiples.
- Deja el texto en una forma más uniforme.

**Interpretación**
La limpieza reduce ruido y mejora tanto la creación de chunks como la calidad de los embeddings. Es un paso sencillo, pero importante para que la indexación no arrastre problemas de formato.

In [ ]:
import re

def clean_text(text):

    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\t+', ' ', text)

    return text.strip()


for doc in documents:
    doc["text"] = clean_text(doc["text"])

## Celda 5 — Definir chunking con overlap

**Objetivo:** dividir el texto en fragmentos manejables.

**Parámetros elegidos**
- `CHUNK_SIZE = 500`
- `CHUNK_OVERLAP = 100`

**Interpretación**
El tamaño del chunk busca un equilibrio entre:
- conservar suficiente contexto,
- y no mezclar demasiadas ideas en un solo fragmento.

El overlap evita que una idea importante quede cortada justo en el borde entre dos chunks consecutivos.

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

## Celda 6 — Generar chunks

**Objetivo:** construir la colección final de fragmentos que usará el RAG.

**Qué hace esta celda**
- Aplica la función de chunking a cada documento.
- Asigna metadatos a cada chunk:
  - `chunk_id`
  - `source_file`
  - `page_num`
  - `chunk_index_in_page`

**Interpretación**
En esta etapa el corpus deja de ser una lista de páginas y pasa a ser una base de fragmentos pequeños y trazables, lista para convertirse en vectores e indexarse.

In [ ]:
chunked_docs = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for idx, chunk in enumerate(chunks):

        chunked_docs.append({
            "chunk_id": len(chunked_docs),
            "source_file": doc["source_file"],
            "page_num": doc["page_num"],
            "chunk_index_in_page": idx,
            "text": chunk
        })

print("Total chunks:", len(chunked_docs))

Total chunks: 2828


# 3️⃣ BLOQUE 3 — EMBEDDINGS + FAISS

## Celda 7 — Preparar textos y metadatos

**Objetivo:** separar el contenido textual de la información de trazabilidad.

**Qué hace esta celda**
- Crea `chunk_texts`, que contiene solo el texto de cada chunk.
- Crea `chunk_metadata`, que conserva fuente, página e índice local.

**Interpretación**
Esta separación es útil porque:
- el texto se usa para generar embeddings,
- y los metadatos se usan para mostrar referencias y fuentes en la recuperación.

In [ ]:
chunk_texts = [doc["text"] for doc in chunked_docs]

chunk_metadata = []

for doc in chunked_docs:

    chunk_metadata.append({
        "chunk_id": doc["chunk_id"],
        "source_file": doc["source_file"],
        "page_num": doc["page_num"],
        "chunk_index_in_page": doc["chunk_index_in_page"]
    })

## Celda 8 — Generar embeddings

**Objetivo:** representar cada chunk como un vector semántico.

**Modelo usado**
- `all-MiniLM-L6-v2`

**Qué hace esta celda**
- Convierte cada fragmento en un embedding numérico.
- Normaliza los vectores para facilitar comparación por similitud.

**Interpretación**
Los embeddings permiten que el sistema recupere contenido por **significado** y no solo por coincidencia exacta de palabras. Esta es la base de la búsqueda semántica.

In [ ]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/89 [00:00<?, ?it/s]

(2828, 384)


## Celda 9 — Construir índice FAISS HNSW

**Objetivo:** crear el vector store del sistema.

**Configuración destacada**
- Índice: `IndexHNSWFlat`
- `M = 32`
- `efConstruction = 40`
- `efSearch = 32`

**Interpretación**
FAISS permite buscar vecinos más cercanos de forma eficiente. El uso de HNSW mejora velocidad y escalabilidad, lo cual es especialmente útil cuando el corpus crece y ya no es viable comparar contra todos los chunks uno por uno.

In [ ]:
!pip install faiss-cpu

import faiss

embedding_dim = chunk_embeddings.shape[1]

M = 32

index = faiss.IndexHNSWFlat(embedding_dim, M)

index.metric_type = faiss.METRIC_INNER_PRODUCT

index.hnsw.efConstruction = 40
index.hnsw.efSearch = 32

index.add(chunk_embeddings.astype("float32"))

print("Vectores indexados:", index.ntotal)

Vectores indexados: 2828


# 4️⃣ BLOQUE 4 — BM25

## Celda 10 — Tokenización

**Objetivo:** preparar el corpus para búsqueda léxica.

**Qué hace esta celda**
- Tokeniza cada chunk en palabras.
- Convierte el texto a minúsculas.

**Interpretación**
A diferencia de FAISS, BM25 no trabaja con vectores semánticos sino con coincidencias de términos. Esta etapa habilita el componente léxico del sistema híbrido.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve the LookupError

from nltk.tokenize import word_tokenize

tokenized_corpus = [word_tokenize(text.lower()) for text in chunk_texts]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Celda 11 — Construir índice BM25

**Objetivo:** habilitar la recuperación por coincidencia textual.

**Qué hace esta celda**
- Crea un objeto `BM25Okapi` a partir del corpus tokenizado.

**Interpretación**
BM25 es útil cuando la consulta contiene términos muy específicos. Complementa a FAISS, que captura mejor relaciones semánticas más flexibles.

In [ ]:
!pip install rank_bm25

from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

## Celda 12 — Función `bm25_search`

**Objetivo:** recuperar chunks relevantes por coincidencia de palabras.

**Qué hace esta función**
- Tokeniza la consulta.
- Calcula scores BM25 para todos los chunks.
- Devuelve los mejores resultados con metadatos.

**Interpretación**
Esta función constituye el motor de búsqueda léxica del sistema. Es especialmente valiosa cuando se quiere priorizar precisión terminológica.

In [ ]:
import numpy as np

def bm25_search(query, top_k=5):

    tokenized_query = word_tokenize(query.lower())

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "score": float(scores[idx]),
            "chunk_id": chunk_metadata[idx]["chunk_id"],
            "source_file": chunk_metadata[idx]["source_file"],
            "page_num": chunk_metadata[idx]["page_num"],
            "text": chunk_texts[idx]
        })

    return results

# 5️⃣ BLOQUE 5 — HYBRID RETRIEVAL

En este bloque se combinan dos enfoques de recuperación:

1. **FAISS**, que recupera por similitud semántica.
2. **BM25**, que recupera por coincidencia textual.

**Interpretación**
La búsqueda híbrida suele ser más robusta que usar solo uno de los métodos, porque aprovecha las fortalezas de ambos.

In [ ]:
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    D, I = index.search(query_embedding, top_k)

    results = []

    for i in range(top_k):

        chunk_id = I[0][i]
        score = D[0][i]

        meta = chunk_metadata[chunk_id]

        results.append({
            "score": float(score),
            "chunk_id": chunk_id,
            "source_file": meta["source_file"],
            "page_num": meta["page_num"],
            "text": chunk_texts[chunk_id]
        })

    return results

## Función `hybrid_search`

**Objetivo:** fusionar resultados de FAISS y BM25.

**Qué hace esta función**
- Obtiene candidatos desde la búsqueda semántica.
- Obtiene candidatos desde la búsqueda léxica.
- Combina ambos scores por `chunk_id`.
- Ordena y devuelve los mejores.

**Interpretación**
Aquí se implementa uno de los requisitos más importantes del proyecto: **Hybrid BM25 + FAISS**. Esto mejora la recuperación cuando una consulta necesita tanto cercanía semántica como coincidencia exacta.

In [ ]:
def hybrid_search(query, top_k=5):

    faiss_results = semantic_search(query, top_k=top_k*2)

    bm25_results = bm25_search(query, top_k=top_k*2)

    combined = {}

    for r in faiss_results:
        combined[r["chunk_id"]] = combined.get(r["chunk_id"],0) + r["score"]

    for r in bm25_results:
        combined[r["chunk_id"]] = combined.get(r["chunk_id"],0) + r["score"]

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    results = []

    for chunk_id, score in ranked[:top_k]:

        meta = chunk_metadata[chunk_id]

        results.append({
            "score": score,
            "chunk_id": chunk_id,
            "source_file": meta["source_file"],
            "page_num": meta["page_num"],
            "text": chunk_texts[chunk_id]
        })

    return results

# 6️⃣ BLOQUE 6 — RERANKER

En este bloque se incorpora una segunda etapa de filtrado.

**Idea principal**
- El retrieval híbrido trae candidatos relevantes.
- El reranker los vuelve a comparar con la consulta.
- Se queda con los chunks mejor alineados.

**Interpretación**
El reranker mejora la precisión fina del contexto final que verá el LLM.

In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker_model = CrossEncoder(RERANKER_MODEL_NAME)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Función `rerank_results`

**Objetivo:** reordenar candidatos con un cross-encoder.

**Qué hace esta función**
- Evalúa pares `(query, chunk)` con el modelo reranker.
- Asigna un score más preciso.
- Devuelve solo los mejores fragmentos.

**Interpretación**
Este paso reduce ruido y mejora la calidad del contexto final, lo que normalmente se traduce en respuestas más relevantes.

In [ ]:
def rerank_results(query, candidates, top_k=4):

    pairs = [[query, c["text"]] for c in candidates]

    scores = reranker_model.predict(pairs)

    for i, s in enumerate(scores):
        candidates[i]["rerank_score"] = float(s)

    ranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return ranked[:top_k]

# 7️⃣ BLOQUE 7 — LLM (Qwen)

En este bloque se carga el modelo generativo que producirá la respuesta final.

**Modelo elegido**
- `Qwen/Qwen2.5-3B-Instruct`

**Interpretación**
Qwen actúa como el componente generador del sistema. Su función no es memorizar el corpus, sino redactar una respuesta apoyándose en el contexto recuperado.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

# 8️⃣ BLOQUE 8 — GENERAR RESPUESTA

**Objetivo:** construir el prompt y producir una respuesta con el LLM.

**Qué hace esta celda**
- Une los chunks recuperados en un solo bloque de contexto.
- Inserta contexto y pregunta en el prompt.
- Genera la respuesta con Qwen.

**Interpretación**
Aquí se materializa el concepto de RAG: el modelo no responde “desde cero”, sino después de leer un contexto recuperado dinámicamente.

In [ ]:
def generate_answer(query, context_chunks):

    context_text = "\n\n".join([c["text"] for c in context_chunks])

    prompt = f"""
Responde usando únicamente la información del contexto.

Contexto:
{context_text}

Pregunta:
{query}

Respuesta:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# 9️⃣ BLOQUE 9 — PIPELINE RAG

**Objetivo:** integrar recuperación + generación en un solo flujo.

**Qué hacen estas funciones**
- `retrieve_context`: ejecuta retrieval híbrido y reranking.
- `rag_pipeline`: ejecuta recuperación y luego generación.

**Interpretación**
Este bloque conecta todos los componentes anteriores en una sola tubería funcional.

In [ ]:
def retrieve_context(query, initial_k=8, final_k=4):

    hybrid_results = hybrid_search(query, top_k=initial_k)

    reranked = rerank_results(query, hybrid_results, top_k=final_k)

    return reranked

In [ ]:
def rag_pipeline(query):

    context_chunks = retrieve_context(query)

    answer = generate_answer(query, context_chunks)

    return answer, context_chunks

# 1️⃣0️⃣ BLOQUE 10 — DEMO

En esta sección se ejecuta una consulta real al sistema.

**Qué se busca mostrar**
- que el pipeline corre de extremo a extremo,
- que se recupera contexto relevante,
- y que el LLM produce una respuesta basada en ese contexto.

In [ ]:
query = "¿Qué es el error de no respuesta en una encuesta?"

answer, context = rag_pipeline(query)

print(answer)


Responde usando únicamente la información del contexto.

Contexto:
ENTARSE EN CAMPO Uno de los problemas más serios en una encuesta de hogare s por muestreo, es la "no - respuesta", esto es, el no lograr obtener información para ciertos hogares o la falta de entrevista a personas elegibles. En muchos casos, los encuestadores deberán realizar visitas a los hogares por la noche o en los fines de se mana, con el fin de reducir la falta de respuesta. Es una tarea que exige tiempo y requiere un seguimiento estricto por medio de las hojas de control de avance diario.

Buenas Prácticas de una Encuesta por Muestreo Instituto Nacional de Estadística e Informática 3 2.1.6 Diseñar un formulario que refleje lo s objetivos de la encuesta, que facilite la capacitación, operación de campo y el procesamiento de la información: • Debe tenerse cuidado en la redacción de las preguntas, • Tener en cuenta el flujo lógico de las preguntas, • Formato amigable para el informante y el entrevistador, • Debe re

## Celda de apoyo — Instalar métricas de evaluación

**Objetivo:** preparar el entorno para medir el desempeño del sistema.

Esta celda instala las librerías necesarias para calcular métricas automáticas, principalmente ROUGE y BLEU.

In [ ]:
!pip install rouge-score nltk

## Celda de apoyo — Importar métricas y utilidades

**Objetivo:** configurar los objetos necesarios para la evaluación.

Aquí se inicializan los evaluadores que luego compararán respuestas generadas contra respuestas de referencia.

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import nltk

nltk.download("punkt")

scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
smooth = SmoothingFunction().method1

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Celda de apoyo — Función `grounding_score`

**Objetivo:** estimar qué tan anclada está la respuesta al contexto recuperado.

**Cómo funciona**
- Tokeniza la respuesta.
- Tokeniza el contexto recuperado.
- Calcula qué fracción de términos de la respuesta aparece también en el contexto.

**Interpretación**
Un grounding alto sugiere que la salida se apoya fuertemente en la evidencia recuperada y no inventa demasiado contenido.

In [ ]:
import re

def grounding_score(answer, context_chunks):

    answer_tokens = set(re.findall(r'\b\w+\b', answer.lower()))

    context_text = " ".join([c["text"] for c in context_chunks])
    context_tokens = set(re.findall(r'\b\w+\b', context_text.lower()))

    if len(answer_tokens) == 0:
        return 0

    overlap = answer_tokens.intersection(context_tokens)

    return len(overlap) / len(answer_tokens)

## Celda de apoyo — Preguntas de evaluación

**Objetivo:** definir un conjunto corto de consultas para validación.

Estas preguntas representan distintos tipos de información del dominio: cobertura, diseño muestral, consistencia y revisión de bases de datos.

In [ ]:
test_questions = [

"¿Qué es el error de cobertura en una encuesta?",

"¿Qué es el diseño muestral?",

"¿Por qué es importante la consistencia de datos en encuestas?",

"¿Por qué es importante revisar la base de datos antes del análisis?"

]

## Celda de apoyo — Respuestas de referencia

**Objetivo:** establecer contra qué se compararán las respuestas generadas.

Estas referencias no reemplazan el juicio cualitativo, pero sirven como base para métricas automáticas como ROUGE y BLEU.

In [ ]:
reference_answers = {

"¿Qué es el error de cobertura en una encuesta?":
"El error de cobertura ocurre cuando no se obtiene información para algunos hogares o cuando ciertas unidades de la población no están incluidas en la encuesta.",

"¿Qué es el diseño muestral?":
"El diseño muestral es el conjunto de procedimientos utilizados para seleccionar las unidades de la muestra en una encuesta.",

"¿Por qué es importante la consistencia de datos en encuestas?":
"La consistencia de los datos permite asegurar que la información recolectada sea coherente y confiable.",

"¿Por qué es importante revisar la base de datos antes del análisis?":
"Revisar la base de datos antes del análisis permite detectar errores, inconsistencias y datos faltantes."
}

## Celda de apoyo — Ejecutar evaluación por pregunta

**Objetivo:** correr el pipeline sobre cada pregunta de prueba y registrar métricas.

**Qué se calcula aquí**
- Respuesta generada por el sistema.
- ROUGE frente a la referencia.
- BLEU frente a la referencia.
- Grounding frente al contexto recuperado.

**Interpretación**
Esta celda produce la evidencia cuantitativa del comportamiento del sistema.

In [ ]:
rouge_scores = []
bleu_scores = []
grounding_scores = []

for q in test_questions:

    respuesta_modelo, context = rag_pipeline(q)

    respuesta_referencia = reference_answers[q]

    # ROUGE
    score = scorer.score(respuesta_referencia, respuesta_modelo)
    rouge_scores.append(score["rouge1"].fmeasure)

    # BLEU
    bleu = sentence_bleu(
        [respuesta_referencia.split()],
        respuesta_modelo.split(),
        smoothing_function=smooth
    )
    bleu_scores.append(bleu)

    # Grounding
    grounding = grounding_score(respuesta_modelo, context)
    grounding_scores.append(grounding)

    print("="*80)
    print("PREGUNTA:", q)
    print("\nRESPUESTA MODELO:")
    print(respuesta_modelo[:400])

PREGUNTA: ¿Qué es el error de cobertura en una encuesta?

RESPUESTA MODELO:

Responde usando únicamente la información del contexto.

Contexto:
 exige tiempo y requiere un seguimiento estricto por medio de las hojas de control de avance diario. Si el nivel de la no -respuesta es muy elevado, se podría producir una distorsión muy i mportante en los resultados de la encuesta. En tal sentido, una de las tareas más importantes del supervisor y encuestador es tratar de reducir
PREGUNTA: ¿Qué es el diseño muestral?

RESPUESTA MODELO:

Responde usando únicamente la información del contexto.

Contexto:
al: anual. De acuerdo al diseño muestral, se podrá producir resultados para diferentes “arreglos” de unidades y su nivel de desagregación dependerá fundamentalmente de la precisión (error de muestreo) con que se estime el dato, y ésta del tamaño de la muestra para cada caso. El tamaño anual de la muestra 2022 es de 36 8 48 viviend
PREGUNTA: ¿Por qué es importante la consistencia de datos en encu

## Celda de apoyo — Resultados promedio

**Objetivo:** resumir el desempeño general del sistema en un solo bloque.

Aquí se muestran promedios de ROUGE, BLEU y Grounding. Este resumen es útil para las diapositivas y para una lectura rápida del rendimiento global.

In [ ]:
print("\n==============================")
print("RESULTADOS PROMEDIO")
print("==============================")

print("ROUGE promedio:", round(np.mean(rouge_scores),2))
print("BLEU promedio:", round(np.mean(bleu_scores),2))
print("Grounding promedio:", round(np.mean(grounding_scores),2))


RESULTADOS PROMEDIO
ROUGE promedio: 0.07
BLEU promedio: 0.01
Grounding promedio: 0.8


## Celda de apoyo — Interpretación automática de resultados

**Objetivo:** convertir métricas numéricas en una lectura más comprensible.

Esta celda ayuda a explicar por qué algunas métricas pueden ser bajas aunque el sistema funcione correctamente; por ejemplo, cuando el modelo parafrasea mucho.

In [ ]:
print("\n==============================")
print("INTERPRETACIÓN DE RESULTADOS")
print("==============================")

rouge_avg = np.mean(rouge_scores)
bleu_avg = np.mean(bleu_scores)
ground_avg = np.mean(grounding_scores)

print(f"ROUGE promedio: {round(rouge_avg,2)}")
print(f"BLEU promedio: {round(bleu_avg,2)}")
print(f"Grounding promedio: {round(ground_avg,2)}")

print("\nInterpretación:")

if ground_avg >= 0.75:
    print("- El sistema muestra un alto nivel de grounding, lo que indica que las respuestas se basan principalmente en el contexto recuperado.")
else:
    print("- El grounding es moderado o bajo, lo que sugiere que el modelo podría generar información fuera del contexto.")

if rouge_avg < 0.2:
    print("- El ROUGE es bajo, lo cual puede deberse a que el modelo reformula las respuestas en lugar de copiar textualmente el contenido del documento.")

if bleu_avg < 0.1:
    print("- El BLEU es bajo porque el modelo generativo produce respuestas parafraseadas, reduciendo la coincidencia exacta de palabras.")

print("\nConclusión:")
print("El sistema RAG logra mantener coherencia con el contexto recuperado, aunque las métricas basadas en coincidencia textual penalizan las respuestas parafraseadas.")


INTERPRETACIÓN DE RESULTADOS
ROUGE promedio: 0.07
BLEU promedio: 0.01
Grounding promedio: 0.8

Interpretación:
- El sistema muestra un alto nivel de grounding, lo que indica que las respuestas se basan principalmente en el contexto recuperado.
- El ROUGE es bajo, lo cual puede deberse a que el modelo reformula las respuestas en lugar de copiar textualmente el contenido del documento.
- El BLEU es bajo porque el modelo generativo produce respuestas parafraseadas, reduciendo la coincidencia exacta de palabras.

Conclusión:
El sistema RAG logra mantener coherencia con el contexto recuperado, aunque las métricas basadas en coincidencia textual penalizan las respuestas parafraseadas.


## Celda de apoyo — Demo final del sistema

**Objetivo:** cerrar el notebook con una prueba visible y fácil de presentar.

La idea es mostrar:
- una pregunta clara,
- la respuesta generada,
- y las fuentes del contexto usadas por el sistema.

In [ ]:
print("\n==============================")
print("DEMO FINAL DEL SISTEMA RAG")
print("==============================")

query_demo = "¿Qué es el error de no respuesta en una encuesta?"

answer, context = rag_pipeline(query_demo)

print("\nPregunta:")
print(query_demo)

print("\nRespuesta generada:")
print(answer)

print("\nFuentes utilizadas:")
for c in context:
    print(f"- {c['source_file']} (página {c['page_num']})")


DEMO FINAL DEL SISTEMA RAG

Pregunta:
¿Qué es el error de no respuesta en una encuesta?

Respuesta generada:

Responde usando únicamente la información del contexto.

Contexto:
ENTARSE EN CAMPO Uno de los problemas más serios en una encuesta de hogare s por muestreo, es la "no - respuesta", esto es, el no lograr obtener información para ciertos hogares o la falta de entrevista a personas elegibles. En muchos casos, los encuestadores deberán realizar visitas a los hogares por la noche o en los fines de se mana, con el fin de reducir la falta de respuesta. Es una tarea que exige tiempo y requiere un seguimiento estricto por medio de las hojas de control de avance diario.

Buenas Prácticas de una Encuesta por Muestreo Instituto Nacional de Estadística e Informática 3 2.1.6 Diseñar un formulario que refleje lo s objetivos de la encuesta, que facilite la capacitación, operación de campo y el procesamiento de la información: • Debe tenerse cuidado en la redacción de las preguntas, • Tener e